# New Jersey — Title 17B (Insurance) → `data/new_jersey/ins_codes/*.md`

New Jersey’s core **insurance** statutes are **Title 17B — Insurance** of the **New Jersey Revised Statutes (N.J.S.A.)**. On **Justia**, sections are linked directly from **[`/codes/new-jersey/title-17b/`](https://law.justia.com/codes/new-jersey/title-17b/)** as **`/section-17b-…`** (no intermediate chapter index pages).

**Note:** **Title 17** (*Corporations and Institutions for Finance and Insurance*) also contains some insurance-related provisions; this notebook only downloads **Title 17B**. Extend similarly if you need Title 17.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** One GET of the Title 17B index; collect every **`section-`** URL under **`/codes/new-jersey/title-17b/`** (~**1,000** sections).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`NJ_sec_<label>.md`** where **`<label>`** is the slug after **`section-`** (e.g. `17b-17-1` → `NJ_sec_17b_17_1.md`). Display citation **`17B:17-1`**.

**Config:** **`MAX_SECTIONS`** caps downloads (**0** = all). **`REUSE_DISCOVERED_URLS`** skips discovery when **`_nj_title17b_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/new-jersey/title-17b"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "new_jersey" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_nj_title17b_section_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def discover_section_urls() -> list[str]:
    """Single index page lists all Title 17B section links."""
    html = curl_get(TITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    sections: set[str] = set()
    for a in soup.find_all("a", href=True):
        absu = urljoin(TITLE_INDEX, a["href"])
        pk = path_key(absu).lower()
        if not pk.startswith(PATH_PREFIX):
            continue
        if "/section-" not in pk:
            continue
        sections.add(absu if absu.endswith("/") else absu + "/")
    return sorted(sections)


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """17b-17-1 -> 17B:17-1 ; 17b-17-5.1 -> 17B:17-5.1"""
    m = re.match(r"^17b-(.+)$", label, re.I)
    if m:
        return f"17B:{m.group(1)}"
    return label


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"NJ_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("N.J. Rev" in s or "New Jersey Rev" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("N.J. Rev. Stat") or s.startswith("NJ Rev Stat"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_17b() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 17B")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"New Jersey Revised Statutes {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**New Jersey Revised Statutes — Title 17B (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official statutes:** [NJ Legislature — Statutes](https://www.njleg.state.nj.us/statutes)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** N.J.S.A. {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_17b()


Discovered 994 section URLs under Title 17B
… 200/994 (wrote=200 skipped=0 failed=0)
… 400/994 (wrote=400 skipped=0 failed=0)
… 600/994 (wrote=600 skipped=0 failed=0)
… 800/994 (wrote=800 skipped=0 failed=0)
Done. wrote=994 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/new_jersey/ins_codes


{'wrote': 994, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
